In [1]:
from dotenv import load_dotenv

load_dotenv()

%load_ext mypy_ipython

In [2]:
from langgraph.graph import START, END, StateGraph, add_messages, MessagesState
from typing_extensions import TypedDict
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.messages import AIMessage, HumanMessage, BaseMessage
from collections.abc import Sequence
from typing import Literal, Annotated

In [3]:
my_list = add_messages(
    [
        HumanMessage(content="Hello!"),
        AIMessage(content="Hi there! How can I help you today?"),
    ],
    [
        HumanMessage(content="What's the weather like today?"),
    ],
)

In [6]:
my_list

[HumanMessage(content='Hello!', additional_kwargs={}, response_metadata={}, id='e87481b4-9047-446f-98dc-a1e2722ef620'),
 AIMessage(content='Hi there! How can I help you today?', additional_kwargs={}, response_metadata={}, id='452d9a4f-73d4-4988-bfa2-563322fa7110'),
 HumanMessage(content="What's the weather like today?", additional_kwargs={}, response_metadata={}, id='ca3e2e15-9082-4ce3-9da1-f5af2e659e37')]

In [4]:
chat = ChatOpenAI(
    model="gpt-4.1-mini", temperature=0, seed=42, max_completion_tokens=100
)

In [5]:
def ask_question(state: MessagesState) -> MessagesState:
    print(f"\n----> Entered ask_question:")
    for msg in state["messages"]:
        msg.pretty_print()

    question = "What is your question?"
    print(question)

    return MessagesState(messages=[AIMessage(question), HumanMessage(content=input())])

In [6]:
def chatbot(state: MessagesState) -> MessagesState:
    print(f"\n----> Entering chatbot:")
    for msg in state["messages"]:
        msg.pretty_print()

    response = chat.invoke(state["messages"])
    response.pretty_print()

    return MessagesState(messages=[response])

In [7]:
def ask_another_question(state: MessagesState) -> MessagesState:
    print(f"\n----> Entered ask_another_question:")
    for msg in state["messages"]:
        msg.pretty_print()

    question = "What is your next question? (yes/no)"
    print(question)

    return MessagesState(messages=[AIMessage(question), HumanMessage(content=input())])

In [8]:
def routing_function(state: MessagesState) -> Literal["ask_question", "__end__"]:
    if state["messages"][-1].content.lower() in ["yes", "y"]:
        return "ask_question"
    else:
        return "__end__"

In [9]:
graph = StateGraph(MessagesState)

In [10]:
graph.add_node("ask_question", ask_question)
graph.add_node("chatbot", chatbot)
graph.add_node("ask_another_question", ask_another_question)

graph.add_edge(START, "ask_question")
graph.add_edge("ask_question", "chatbot")
graph.add_edge("chatbot", "ask_another_question")
graph.add_conditional_edges(
    source="ask_another_question",
    path=routing_function,
    path_map={"ask_question": "ask_question", "__end__": END},
)

In [11]:
graph_compiled = graph.compile()

In [13]:
graph_compiled.invoke(MessagesState(messages=[]))


----> Entered ask_question:
What is your question?

----> Entering chatbot:
================================== Ai Message ==================================

What is your question?
================================ Human Message =================================

how old is the earth?
================================== Ai Message ==================================

The Earth is approximately 4.54 billion years old. This estimate is based on evidence from radiometric age dating of meteorite material and Earth rocks, as well as lunar samples.

----> Entered ask_another_question:
================================== Ai Message ==================================

What is your question?
================================ Human Message =================================

how old is the earth?
================================== Ai Message ==================================

The Earth is approximately 4.54 billion years old. This estimate is based on evidence from radiometric age dating of meteorit

{'messages': [AIMessage(content='What is your question?', additional_kwargs={}, response_metadata={}, id='f4865b7d-1ad3-4b75-bbd5-f42700fb34e1'),
  HumanMessage(content='how old is the earth?', additional_kwargs={}, response_metadata={}, id='5ef7f751-9b47-4578-98f6-9756f16e1ed7'),
  AIMessage(content='The Earth is approximately 4.54 billion years old. This estimate is based on evidence from radiometric age dating of meteorite material and Earth rocks, as well as lunar samples.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 22, 'total_tokens': 59, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_376a7ccef1', 'id': 'chatcmpl-CswLFV6o3BbtwdytU0zD4y394hLiO', 'service_tier': 'default', 'finish_reason': 